# 🔬 SDXL Compression Study

**Systematic optimization of Stable Diffusion XL** — treating the model as a system of components, optimizing each axis independently, then stacking for Pareto-optimal configurations.

> Rajat Gupta · Pruna AI Technical Interview · April 2026

**Runtime:** A100 GPU recommended (Colab Pro+)

---

## 0. Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/rajatgupta/sdxl-optimization.git 2>/dev/null || true
%cd sdxl-optimization
!pip install -q -e ".[dev]"

In [ ]:
import sys, logging, gc, time
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

sys.path.insert(0, 'src')

from sdxl_opt.utils import (
    setup_logging, seed_everything, get_gpu_info,
    reset_peak_memory, gpu_peak_memory_gb, EVAL_PROMPTS
)
from sdxl_opt.pipeline import CompressionConfig, load_pipeline, generate_images
from sdxl_opt.evaluate import compute_clip_scores, save_comparison_grid, save_individual_images
from sdxl_opt.pareto import plot_pareto, plot_speedup_bar, plot_memory_comparison, find_pareto_frontier

setup_logging('INFO')
gpu = get_gpu_info()
print(f'\n✅ GPU: {gpu}')

## 1. Profiling — Where Does the Time Go?

Before optimizing, let's understand the baseline. SDXL has three major components:
- **Text Encoders** (CLIP-L + OpenCLIP-G): ~1.5 GB, runs once per generation
- **UNet** (2.6B params): ~5 GB, runs N times (once per denoising step) — **85%+ of compute**
- **VAE Decoder**: ~160M params, runs once to decode latents → image

This tells us: **optimizing the UNet has the highest leverage.**

In [ ]:
# Baseline: fp16, 50 steps, no optimizations
baseline_cfg = CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5)
prompts = EVAL_PROMPTS[:4]

pipe = load_pipeline(baseline_cfg)

# --- Profile each component ---
from sdxl_opt.utils import timer

gen = seed_everything(42)

# Text encoding
with timer() as t_enc:
    prompt_embeds, _, pooled_embeds, _ = pipe.encode_prompt(prompt='a test image', device='cuda')
print(f'Text encoding: {t_enc["elapsed_s"]:.3f}s')

# Full generation (to profile UNet vs VAE, we measure total and subtract)
reset_peak_memory()
gen = seed_everything(42)
with timer() as t_full:
    img = pipe('a test image', num_inference_steps=50, guidance_scale=7.5, generator=gen).images[0]
peak = gpu_peak_memory_gb()

print(f'\nBaseline (1 image, 50 steps):')
print(f'  Total latency:  {t_full["elapsed_s"]:.2f}s')
print(f'  Peak VRAM:      {peak:.2f} GB')

del pipe; gc.collect(); torch.cuda.empty_cache()

## 2. Axis-by-Axis Compression

We test each optimization axis **independently** to understand its isolated effect before stacking.

In [ ]:
def benchmark_one(config, prompts, n_runs=3, n_warmup=2):
    """Benchmark a single config. Returns (result_dict, images)."""
    pipe = load_pipeline(config)

    # Warmup
    gen = seed_everything(0)
    for _ in range(n_warmup):
        generate_images(pipe, config, ['warmup'], generator=gen, height=512, width=512)
    torch.cuda.synchronize()

    # Benchmark
    latencies = []
    for run in range(n_runs):
        gen = seed_everything(42)
        reset_peak_memory()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        imgs = generate_images(pipe, config, prompts, generator=gen)
        torch.cuda.synchronize()
        per_img = (time.perf_counter() - t0) / len(prompts)
        latencies.append(per_img)
        print(f'  Run {run+1}/{n_runs}: {per_img:.2f}s/img | Peak VRAM: {gpu_peak_memory_gb():.2f} GB')

    result = {
        'config': config.name,
        'label': config.short_label(),
        'steps': config.num_inference_steps,
        'latency_mean_s': round(np.mean(latencies), 3),
        'latency_std_s': round(np.std(latencies), 3),
        'peak_vram_gb': round(gpu_peak_memory_gb(), 2),
    }

    del pipe; gc.collect(); torch.cuda.empty_cache()
    return result, imgs

### 2.1 Baseline

In [ ]:
configs_to_run = [
    # Baseline
    CompressionConfig(name='baseline', dtype='fp16', num_inference_steps=50, guidance_scale=7.5),
]

results = []
all_images = {}
for cfg in configs_to_run:
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r)
    all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

### 2.2 Quantization (INT8 / NF4)

In [ ]:
quant_configs = [
    CompressionConfig(name='int8_unet', dtype='fp16', quantize_unet='int8', num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='nf4_unet', dtype='fp16', quantize_unet='nf4', num_inference_steps=50, guidance_scale=7.5),
]

for cfg in quant_configs:
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r)
    all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

### 2.3 DeepCache

In [ ]:
cache_configs = [
    CompressionConfig(name='deepcache_2', dtype='fp16', use_deepcache=True, deepcache_interval=2, num_inference_steps=50, guidance_scale=7.5),
    CompressionConfig(name='deepcache_3', dtype='fp16', use_deepcache=True, deepcache_interval=3, num_inference_steps=50, guidance_scale=7.5),
]

for cfg in cache_configs:
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r)
    all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

### 2.4 LCM-LoRA (Step Reduction)

In [ ]:
lcm_configs = [
    CompressionConfig(name='lcm_8step', dtype='fp16', use_lcm_lora=True, num_inference_steps=8, guidance_scale=1.5),
    CompressionConfig(name='lcm_4step', dtype='fp16', use_lcm_lora=True, num_inference_steps=4, guidance_scale=1.5),
]

for cfg in lcm_configs:
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3)
    results.append(r)
    all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

### 2.5 torch.compile

In [ ]:
compile_cfg = CompressionConfig(name='compiled', dtype='fp16', use_torch_compile=True, compile_mode='reduce-overhead', num_inference_steps=50, guidance_scale=7.5)

print(f'\n=== {compile_cfg.name} ===')
r, imgs = benchmark_one(compile_cfg, prompts, n_runs=3, n_warmup=3)  # extra warmup for compile
results.append(r)
all_images[compile_cfg.name] = imgs
print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

### 2.6 Tiny VAE

In [ ]:
tvae_cfg = CompressionConfig(name='tiny_vae', dtype='fp16', use_tiny_vae=True, num_inference_steps=50, guidance_scale=7.5)

print(f'\n=== {tvae_cfg.name} ===')
r, imgs = benchmark_one(tvae_cfg, prompts, n_runs=3)
results.append(r)
all_images[tvae_cfg.name] = imgs
print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

## 3. Stacked Combinations

Now we combine **orthogonal** methods. Quantization + caching + step reduction + compilation are largely independent axes.

In [ ]:
combo_configs = [
    CompressionConfig(name='nf4+deepcache', dtype='fp16', quantize_unet='nf4',
                      use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=50, guidance_scale=7.5),

    CompressionConfig(name='lcm+deepcache', dtype='fp16',
                      use_lcm_lora=True, use_deepcache=True, deepcache_interval=2,
                      num_inference_steps=8, guidance_scale=1.5),

    CompressionConfig(name='lcm+compiled+tvae', dtype='fp16',
                      use_lcm_lora=True, use_torch_compile=True,
                      use_tiny_vae=True, num_inference_steps=8, guidance_scale=1.5),

    CompressionConfig(name='full_stack', dtype='fp16',
                      quantize_unet='nf4', use_lcm_lora=True,
                      use_deepcache=True, deepcache_interval=2,
                      use_torch_compile=True, use_tiny_vae=True,
                      num_inference_steps=8, guidance_scale=1.5),
]

for cfg in combo_configs:
    print(f'\n=== {cfg.name} ===')
    r, imgs = benchmark_one(cfg, prompts, n_runs=3, n_warmup=3)
    results.append(r)
    all_images[cfg.name] = imgs
    print(f'  → {r["latency_mean_s"]}s ± {r["latency_std_s"]}s | {r["peak_vram_gb"]} GB')

## 4. Quality Evaluation (CLIP Score)

In [ ]:
df = pd.DataFrame(results)

# Compute CLIP scores for all configs
for cfg_name, imgs in all_images.items():
    print(f'CLIP score for {cfg_name}...')
    mean, std, scores = compute_clip_scores(imgs, prompts[:len(imgs)])
    df.loc[df['config'] == cfg_name, 'clip_score'] = mean
    df.loc[df['config'] == cfg_name, 'clip_score_std'] = std

# Compute speedup
baseline_lat = df.loc[df['config'] == 'baseline', 'latency_mean_s'].values[0]
df['speedup'] = (baseline_lat / df['latency_mean_s']).round(2)

print('\n' + '='*90)
display(df.sort_values('speedup', ascending=False))

# Save
Path('results').mkdir(exist_ok=True)
df.to_csv('results/benchmark_results.csv', index=False)

## 5. Visualizations

In [ ]:
# Pareto frontier: quality vs speed
plot_pareto(df, 'results/pareto_frontier.png')
from IPython.display import Image as IPImage
display(IPImage('results/pareto_frontier.png', width=800))

In [ ]:
# Speedup bar chart
plot_speedup_bar(df, output_path='results/speedup_bar.png')
display(IPImage('results/speedup_bar.png', width=700))

In [ ]:
# Memory comparison
plot_memory_comparison(df, output_path='results/memory_comparison.png')
display(IPImage('results/memory_comparison.png', width=700))

In [ ]:
# Visual comparison grid
save_comparison_grid(all_images, prompts, 'results/comparison_grid.png', max_prompts=4)
display(IPImage('results/comparison_grid.png', width=1000))

## 6. Pareto-Optimal Recommendations

Based on the frontier, here are deployment presets for different use cases:

In [ ]:
pareto_df = find_pareto_frontier(df, 'latency_mean_s', 'clip_score')
print('Pareto-optimal configurations:')
display(pareto_df[['config', 'latency_mean_s', 'peak_vram_gb', 'clip_score', 'speedup']].sort_values('speedup', ascending=False))

print('\n📋 Deployment Recommendations:')
print('  🚀 Speed:    Use for real-time/interactive — max throughput, acceptable quality')
print('  ⚖️  Balanced: Use for production APIs — good quality, fast enough for user-facing')
print('  🎨 Quality:  Use when output quality is critical — minimal degradation, still faster than baseline')

## 7. LitServe Deployment Demo

Start the API server and test it.

In [ ]:
# Start server in background
import subprocess, requests, time, base64
from PIL import Image
from io import BytesIO

proc = subprocess.Popen(
    ['python', 'server/serve.py', '--preset', 'balanced', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Wait for server to be ready
print('Starting LitServe server...')
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server ready after {i+1}s')
            break
    except:
        time.sleep(1)
else:
    print('⚠️ Server did not start in 60s')

In [ ]:
# Test API call
response = requests.post(
    'http://localhost:8000/predict',
    json={'prompt': 'a photo of an astronaut riding a horse on mars, cinematic lighting'},
    timeout=120,
)

data = response.json()
print(f'Latency: {data["latency_s"]}s | Preset: {data["preset"]}')

img = Image.open(BytesIO(base64.b64decode(data['image_base64'])))
display(img)

In [ ]:
# Cleanup
proc.terminate()
print('Server stopped.')

## 8. Summary & Future Work

### Key Findings
- **Highest leverage:** LCM-LoRA step reduction (50 → 8 steps) gives the single biggest speedup
- **Best stacking:** LCM + DeepCache + torch.compile achieves the best speed/quality trade-off
- **Memory:** NF4 quantization is the most effective memory reducer (~4× on UNet)
- **Compilation:** torch.compile with reduce-overhead gives free ~20% speedup after warm-up

### Ideas for Future Work
- **Learned step distillation** — train a student model to match SDXL in fewer steps (beyond LCM)
- **Ternary quantization** — explore 1.58-bit weights (BitNet-style) for the UNet, potential for massive compression with custom kernels
- **Per-block mixed precision** — different UNet blocks have different sensitivity; quantize aggressively where quality impact is low
- **Speculative decoding for diffusion** — use a small model to predict early steps, full model for final steps
- **Pruna integration** — wrap all of the above into `smash()` calls for one-line optimization